# Lecture 17 · Decoder-only Transformer 문자 생성

Lecture 14와 같은 Shakespeare 문자 데이터를 사용하지만, LSTM 대신 작은 Decoder-only Transformer를 학습합니다.

표준적인 언어모델 방식으로 각 입력 위치에서 **바로 다음 문자**를 예측합니다.

```text
입력 X:  T h e ...
정답 y:  h e   ...
```

## 1. 라이브러리와 설정

In [ ]:
!pip install -q koreanize-matplotlib

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import koreanize_matplotlib

from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

SEQ_LEN = 20
D_MODEL = 32
N_HEADS = 2
FF_DIM = 64
N_LAYERS = 1
DROPOUT = 0.1
EPOCHS = 10
BATCH_SIZE = 64
MAX_CHARS = 5000

assert D_MODEL % N_HEADS == 0


## 2. Shakespeare 문자 데이터 읽기

In [ ]:
path = keras.utils.get_file(
    "shakespeare.txt",
    "https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt"
)

text = open(path, encoding="utf-8").read()[:MAX_CHARS]
chars = sorted(set(text))

char_to_id = {char: idx for idx, char in enumerate(chars)}
id_to_char = {idx: char for char, idx in char_to_id.items()}

vocab_size = len(chars)
encoded = np.array(
    [char_to_id[char] for char in text],
    dtype=np.int32
)

print("전체 문자 수:", len(text))
print("고유 문자 수:", vocab_size)
print("문자 예시:", chars[:20])


## 3. 모든 위치의 입력–정답 쌍 만들기

+입력과 정답은 한 칸 이동한 길이 `SEQ_LEN`의 시퀀스입니다.

In [ ]:
X = np.array([
    encoded[i:i + SEQ_LEN]
    for i in range(len(encoded) - SEQ_LEN)
])

y = np.array([
    encoded[i + 1:i + SEQ_LEN + 1]
    for i in range(len(encoded) - SEQ_LEN)
])

print("X 크기:", X.shape)
print("y 크기:", y.shape)


In [ ]:
input_text = "".join(id_to_char[idx] for idx in X[0])
target_text = "".join(id_to_char[idx] for idx in y[0])

print("입력 X:")
print(repr(input_text))
print("\n정답 y:")
print(repr(target_text))


## 4. Sin/Cos Positional Encoding 만들기

In [ ]:
def build_sinusoidal_pe(seq_len, d_model):
    positions = np.arange(seq_len)[:, np.newaxis]
    dimensions = np.arange(d_model)[np.newaxis, :]

    angles = positions / np.power(
        10000,
        (2 * (dimensions // 2)) / d_model
    )

    angles[:, 0::2] = np.sin(angles[:, 0::2])
    angles[:, 1::2] = np.cos(angles[:, 1::2])

    return angles.astype("float32")

pe_matrix = build_sinusoidal_pe(SEQ_LEN, D_MODEL)
print("PE 크기:", pe_matrix.shape)


## 5. Decoder-only Transformer 만들기

각 Transformer block은 다음 순서로 구성됩니다.

1. causal Self-Attention
2. 잔차 연결과 Layer Normalization
3. Feed-Forward Network
4. 잔차 연결과 Layer Normalization

문자 시퀀스는 길이가 모두 같으므로 padding mask는 필요하지 않습니다.

In [ ]:
token_input = keras.Input(
    shape=(SEQ_LEN,),
    dtype="int32",
    name="characters"
)

x = layers.Embedding(
    input_dim=vocab_size,
    output_dim=D_MODEL,
    name="character_embedding"
)(token_input)

x = keras.ops.add(x, pe_matrix)

for block_index in range(N_LAYERS):
    attention_output = layers.MultiHeadAttention(
        num_heads=N_HEADS,
        key_dim=D_MODEL // N_HEADS,
        dropout=DROPOUT,
        name=f"attention_{block_index + 1}"
    )(
        x,
        x,
        use_causal_mask=True
    )

    attention_output = layers.Dropout(DROPOUT)(attention_output)
    x = layers.Add()([x, attention_output])
    x = layers.LayerNormalization(
        epsilon=1e-6,
        name=f"attention_norm_{block_index + 1}"
    )(x)

    ffn_output = layers.Dense(
        FF_DIM,
        activation="relu",
        name=f"ffn_hidden_{block_index + 1}"
    )(x)
    ffn_output = layers.Dense(
        D_MODEL,
        name=f"ffn_output_{block_index + 1}"
    )(ffn_output)
    ffn_output = layers.Dropout(DROPOUT)(ffn_output)

    x = layers.Add()([x, ffn_output])
    x = layers.LayerNormalization(
        epsilon=1e-6,
        name=f"ffn_norm_{block_index + 1}"
    )(x)

# 마지막 위치만 자르지 않고 모든 위치에서 다음 문자 logits를 출력합니다.
output = layers.Dense(
    vocab_size,
    name="next_character"
)(x)

model = keras.Model(token_input, output)
model.summary()


## 6. 모든 위치의 다음 문자 학습하기

In [ ]:
model.compile(
    optimizer="adam",
    loss=keras.losses.SparseCategoricalCrossentropy(
        from_logits=True
    ),
    metrics=["sparse_categorical_accuracy"]
)

history = model.fit(
    X,
    y,
    validation_split=0.1,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    shuffle=True,
    verbose=2
)


## 7. 학습 곡선 확인하기

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3))

axes[0].plot(history.history["loss"], label="train")
axes[0].plot(history.history["val_loss"], label="validation")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(
    history.history["sparse_categorical_accuracy"],
    label="train"
)
axes[1].plot(
    history.history["val_sparse_categorical_accuracy"],
    label="validation"
)
axes[1].set_title("Next-character Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 8. 다음 문자 예측 확인하기

+모델 출력은 `(batch, sequence, vocabulary)`입니다. 각 위치에서 가장 확률이 높은 다음 문자를 확인합니다.

In [ ]:
logits = model.predict(X[:1], verbose=0)
predicted_ids = logits.argmax(axis=-1)[0]

print("출력 크기:", logits.shape)
print()

for position in range(8):
    current_char = id_to_char[X[0, position]]
    true_next = id_to_char[y[0, position]]
    predicted_next = id_to_char[predicted_ids[position]]

    print(
        repr(current_char),
        "→ 정답:", repr(true_next),
        "/ 예측:", repr(predicted_next)
    )


## 9. 마지막 위치의 확률로 문자를 반복 생성하기

+학습에는 모든 위치를 사용하지만, 생성할 때는 현재 문맥의 마지막 위치에서 다음 문자를 선택합니다.

In [ ]:
def generate_text(seed, length=200, temperature=1.0):
    if temperature <= 0:
        raise ValueError("temperature는 0보다 커야 합니다.")

    if len(seed) < SEQ_LEN:
        raise ValueError(f"seed는 {SEQ_LEN}자 이상이어야 합니다.")

    unknown_chars = [char for char in seed if char not in char_to_id]
    if unknown_chars:
        raise ValueError(f"학습 어휘에 없는 문자가 있습니다: {unknown_chars}")

    sequence = [char_to_id[char] for char in seed]
    generated_chars = []

    for _ in range(length):
        context = np.array(
            [sequence[-SEQ_LEN:]],
            dtype=np.int32
        )

        logits = model.predict(context, verbose=0)[0, -1]
        scaled_logits = logits / temperature
        probabilities = tf.nn.softmax(scaled_logits).numpy()

        next_id = np.random.choice(
            vocab_size,
            p=probabilities
        )

        sequence.append(next_id)
        generated_chars.append(id_to_char[next_id])

    return "".join(generated_chars)


In [ ]:
seed = text[:SEQ_LEN]

print("Seed:")
print(repr(seed))

for temperature in [0.5, 1.0, 1.5]:
    generated = generate_text(
        seed,
        length=200,
        temperature=temperature
    )

    print(f"\n[Temperature = {temperature}]")
    print(seed + generated)


## 10. Lecture 14와 비교

- Lecture 14 LSTM은 은닉 상태를 순서대로 갱신합니다.
- Lecture 17 Transformer는 causal mask 안에서 이전 위치들을 Self-Attention으로 참고합니다.
- 두 모델 모두 문자 단위 다음 문자 예측과 반복 생성을 수행합니다.
- 작은 데이터와 짧은 학습은 자연스러운 문장 생성보다 전체 생성 과정을 이해하는 데 목적이 있습니다.